# Multi-Echelon Supply Chain Network Optimization Engine
### Operations Research | Mixed-Integer Linear Programming (MILP) | Network Flow Optimization

This notebook formulates and solves an exact **3-Echelon Mixed-Integer Linear Program (MILP)** to minimize total network costs across:
- **5 Raw Material Suppliers** (Supplying Materials A, B, C, D)
- **3 Manufacturing Assembly Plants** (Converting materials to finished goods under multi-material BOM constraints)
- **4 Regional Customer Markets** (Fulfilling 100% of weekly customer demands across Products 1, 2, 3, 4)

In [1]:
import pandas as pd
import numpy as np
import pulp
import os

# 1. Ingest Multi-Echelon Network Topology and Benchmark Parameters
data_path = os.path.join("data", "supply_chain_benchmark.xlsx")
if not os.path.exists(data_path):
    data_path = "supply_chain_benchmark.xlsx"

sup_stock = pd.read_excel(data_path, sheet_name="Supplier stock", index_col=0).fillna(0)
raw_mat_cost = pd.read_excel(data_path, sheet_name="Raw material costs", index_col=0).fillna(0)
raw_mat_ship = pd.read_excel(data_path, sheet_name="Raw material shipping", index_col=0).fillna(0)
bom = pd.read_excel(data_path, sheet_name="Product requirements", index_col=0).fillna(0)
prod_cap = pd.read_excel(data_path, sheet_name="Production capacity", index_col=0).fillna(0)
demand = pd.read_excel(data_path, sheet_name="Customer demand", index_col=0).fillna(0)
prod_cost = pd.read_excel(data_path, sheet_name="Production cost", index_col=0).fillna(0)
ship_costs = pd.read_excel(data_path, sheet_name="Shipping costs", index_col=0).fillna(0)

suppliers = list(sup_stock.index)
raw_materials = list(sup_stock.columns)
factories = list(prod_cap.index)
products = list(prod_cost.columns)
customers = list(demand.columns)

total_demand_units = float(demand.values.sum())
print(f"Suppliers: {len(suppliers)} | Raw Materials: {len(raw_materials)}")
print(f"Plants: {len(factories)} | Products: {len(products)} | Regional Customers: {len(customers)}")
print(f"Total Weekly Customer Demand: {total_demand_units:,.0f} units")

Suppliers: 5 | Raw Materials: 4
Plants: 3 | Products: 4 | Regional Customers: 4
Total Weekly Customer Demand: 7,500 units


## 2. Mathematical Formulation & Exact MILP Optimization

### Objective Function:
$$\min \sum_{f,m,s} \text{orders}_{fms} \cdot (\text{MatCost}_{sm} + \text{InboundRate}_{sf}) + \sum_{f,p} \text{vol}_{fp} \cdot \text{ProdCost}_{fp} + \sum_{f,c,p} \text{delivery}_{fcp} \cdot \text{OutboundRate}_{fc}$$

### Operational Constraints:
1. **Supplier Availability Limits**: $\sum_{f} \text{orders}_{fms} \le \text{SupplierStock}_{sm} \quad \forall s, m$
2. **Plant Throughput Capacity**: $\sum_{p} \text{vol}_{fp} \le \text{PlantCapacity}_{f} \quad \forall f$
3. **Bill of Materials (BOM) Balance**: $\sum_{s} \text{orders}_{fms} \ge \sum_{p} \text{vol}_{fp} \cdot \text{BOM}_{pm} \quad \forall f, m$
4. **Plant Flow Conservation**: $\text{vol}_{fp} \ge \sum_{c} \text{delivery}_{fcp} \quad \forall f, p$
5. **100% Demand Satisfaction**: $\sum_{f} \text{delivery}_{fcp} \ge \text{Demand}_{pc} \quad \forall c, p$

In [3]:
# Initialize Mixed-Integer Linear Programming Model
prob = pulp.LpProblem("Multi_Echelon_Supply_Chain_Network_Optimization", pulp.LpMinimize)

# Decision Variables
orders = pulp.LpVariable.dicts("Order", [(f, m, s) for f in factories for m in raw_materials for s in suppliers], lowBound=0)
vol = pulp.LpVariable.dicts("ProdVol", [(f, p) for f in factories for p in products], lowBound=0)
delivery = pulp.LpVariable.dicts("Delivery", [(f, c, p) for f in factories for c in customers for p in products], lowBound=0)

# Objective Function: Min Procurement + Manufacturing + Distribution
prob += (
    pulp.lpSum(orders[f, m, s] * (raw_mat_cost.loc[s, m] + raw_mat_ship.loc[s, f]) for f in factories for m in raw_materials for s in suppliers)
    + pulp.lpSum(vol[f, p] * prod_cost.loc[f, p] for f in factories for p in products)
    + pulp.lpSum(delivery[f, c, p] * ship_costs.loc[f, c] for f in factories for c in customers for p in products)
)

# 1. Supplier Capacity Limits
for s in suppliers:
    for m in raw_materials:
        prob += pulp.lpSum(orders[f, m, s] for f in factories) <= float(sup_stock.loc[s, m])

# 2. Plant Manufacturing Capacity Bounds
for f in factories:
    prob += pulp.lpSum(vol[f, p] for p in products) <= float(prod_cap.loc[f, 'Capacity'])

# 3. Multi-Material Bill of Materials (BOM) Conversion Balance
for f in factories:
    for m in raw_materials:
        prob += pulp.lpSum(orders[f, m, s] for s in suppliers) >= pulp.lpSum(vol[f, p] * float(bom.loc[p, m]) for p in products)

# 4. Plant Flow Conservation
for f in factories:
    for p in products:
        prob += vol[f, p] >= pulp.lpSum(delivery[f, c, p] for c in customers)

# 5. Customer Demand Satisfaction (100% Fulfillment)
for c in customers:
    for p in products:
        prob += pulp.lpSum(delivery[f, c, p] for f in factories) >= float(demand.loc[p, c])

# Solve MILP
prob.solve(pulp.PULP_CBC_CMD(msg=0))

total_optimal_cost = float(pulp.value(prob.objective))
procure_cost = float(sum(orders[f, m, s].varValue * (raw_mat_cost.loc[s, m] + raw_mat_ship.loc[s, f]) for f in factories for m in raw_materials for s in suppliers if orders[f, m, s].varValue))
manuf_cost = float(sum(vol[f, p].varValue * prod_cost.loc[f, p] for f in factories for p in products if vol[f, p].varValue))
distrib_cost = float(sum(delivery[f, c, p].varValue * ship_costs.loc[f, c] for f in factories for c in customers for p in products if delivery[f, c, p].varValue))
total_produced = float(sum(vol[f, p].varValue for f in factories for p in products if vol[f, p].varValue))

print("=" * 85)
print(f"Optimization Status        : {pulp.LpStatus[prob.status]}")
print(f"Total Optimal Network Cost : ${total_optimal_cost:,.2f}")
print(f"Raw Material Procurement   : ${procure_cost:,.2f} ({procure_cost/total_optimal_cost*100:.1f}%)")
print(f"Plant Manufacturing Cost   : ${manuf_cost:,.2f} ({manuf_cost/total_optimal_cost*100:.1f}%)")
print(f"Outbound Distribution Cost : ${distrib_cost:,.2f} ({distrib_cost/total_optimal_cost*100:.1f}%)")
print(f"Total Volume Produced      : {total_produced:,.0f} units (Demand: {total_demand_units:,.0f} units)")
print(f"Demand Fulfillment Rate    : {(total_produced/total_demand_units)*100:.1f}%")
print("=" * 85)

Optimization Status        : Optimal
Total Optimal Network Cost : $986,882.96
Raw Material Procurement   : $839,235.23 (85.0%)
Plant Manufacturing Cost   : $121,486.36 (12.3%)
Outbound Distribution Cost : $26,161.36 (2.7%)
Total Volume Produced      : 7,500 units (Demand: 7,500 units)
Demand Fulfillment Rate    : 100.0%
